# Agente IA con Historial de Conversación en PostgreSQL

> Extiende el agente básico añadiendo **memoria persistente**: cada mensaje se guarda en PostgreSQL y puede recuperarse por `session_id`.

### Requisitos antes de ejecutar este notebook:

* Contar con API key de OpenAI (https://platform.openai.com)
* Contar con un proyecto en Supabase (https://supabase.com/) y obtener credenciales de conexión a la DB donde se guardarán las conversaciones Usuario-Bot:
    - DB_USER
    - DB_PASSWORD
    - DB_HOST
    - DB_PORT
    - DB_NAME


---

## ¿Qué cambia respecto al agente básico?

| | Agente Básico | Este Agente |
|---|---|---|
| Memoria | ❌ Stateless — sin memoria | ✅ Persistente en PostgreSQL |
| Prompt | `SystemMessage + HumanMessage` | `SystemMessage + MessagesPlaceholder + HumanMessage` |
| Chain | `prompt \| chat` | `RunnableWithMessageHistory(prompt \| chat)` |
| Sesiones | No aplica | Identificadas por `session_id` (UUID) |
| Base de datos | No usa | PostgreSQL — tabla `tbl_chat_history_text` |

---

## Los 4 momentos clave del historial

| # | Momento | ¿Dónde ocurre? | ¿Qué hace? |
|---|---|---|---|
| 1 | **Historial LEÍDO** | `get_session_history(session_id)` | Conecta a PostgreSQL y **recupera todos los mensajes de la sesión** |
| 2 | **Historial INYECTADO** | `MessagesPlaceholder(variable_name="history")` | Inserta los mensajes pasados en el prompt antes de llamar al LLM |
| 3 | **Respuesta GENERADA** | `GPT-4.1` via OpenAI API | Recibe system + historial + mensaje actual y genera la respuesta |
| 4 | **Historial GUARDADO** | `RunnableWithMessageHistory` (automático) | Guarda HumanMessage + AIMessage en PostgreSQL al terminar |

## Arquitectura

> Se renderiza automáticamente en **GitHub**. En JupyterLab requiere `pip install jupyterlab-mermaid`.

```mermaid
flowchart TD
    U(["Usuario\ninvoke: input + session_id"])

    subgraph RWMH["RunnableWithMessageHistory · langchain_core.runnables.history"]
        direction TB
        GH["① get_session_history(session_id)\nPostgresChatMessageHistory\nlee historial desde PostgreSQL"]

        subgraph LCEL["Cadena LCEL — prompt | chat"]
            direction TB
            P["② ChatPromptTemplate · langchain_core.prompts\n────────────────────────────────\nSystemMessage       → rol del bot\nMessagesPlaceholder → historial inyectado aquí\nHumanMessage        → mensaje actual"]
            M["③ GPT-4.1 · OpenAI API · langchain.chat_models\ninit_chat_model · temperature = 0.7"]
        end
    end

    DB[("④ PostgreSQL\ntbl_chat_history_text\nhistorial por session_id")]
    R(["respuesta.content\nTexto final al usuario"])

    U     -->|"invoke"| RWMH
    GH    -->|"historial previo"| P
    P     -->|"System + Historial + Human"| M
    M     -->|"AIMessage"| R
    GH   <-->|"lee mensajes"| DB
    RWMH  -->|"④ guarda Human + AI"| DB

    style U    fill:#1976D2,stroke:#0D47A1,color:#fff
    style GH   fill:#7B1FA2,stroke:#4A148C,color:#fff
    style P    fill:#F57C00,stroke:#E65100,color:#fff
    style M    fill:#388E3C,stroke:#1B5E20,color:#fff
    style DB   fill:#C62828,stroke:#B71C1C,color:#fff
    style R    fill:#1976D2,stroke:#0D47A1,color:#fff
    style RWMH fill:#f3e5f5,stroke:#9C27B0,stroke-dasharray:5 5
    style LCEL fill:#e8f5e9,stroke:#4CAF50,stroke-dasharray:3 3
```

---

### Componentes utilizados

| Componente | Librería | Rol |
|---|---|---|
| `ChatPromptTemplate` | `langchain_core.prompts` | Estructura los mensajes (system + historial + human) |
| `MessagesPlaceholder` | `langchain_core.prompts` | Ranura donde se inyecta el historial en el prompt |
| `init_chat_model` | `langchain.chat_models` | Inicializa GPT-4.1 como modelo de chat |
| `PostgresChatMessageHistory` | `langchain_postgres` | Lee y escribe el historial por `session_id` en PostgreSQL |
| `RunnableWithMessageHistory` | `langchain_core.runnables.history` | Gestiona el historial automáticamente alrededor de la cadena |
| `psycopg` | `psycopg` | Driver sincrónico de conexión a PostgreSQL |

## Imports

Las importaciones clave respecto al agente básico son:
- `MessagesPlaceholder` — nuevo en el prompt para inyectar el historial
- `RunnableWithMessageHistory` — envuelve la cadena para gestionar historial automáticamente
- `PostgresChatMessageHistory` — interfaz con PostgreSQL para leer/escribir mensajes
- `psycopg` — driver de conexión sincrónica a PostgreSQL

In [ ]:
import os

# uuid módulo para generar identificadores únicos universales (UUIDs)
import uuid

from urllib.parse import quote_plus
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_postgres import PostgresChatMessageHistory
import psycopg

In [1]:
# ===========================================
# Se asegura que se carguen las variables de entorno desde el archivo .env
# ===========================================
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

## 1. Configuración de Base de Datos

Las credenciales se leen de variables de entorno definidas en el archivo `.env` de la raíz del proyecto.

| Variable | Requerida | Default | Descripción |
|---|---|---|---|
| `DB_USER` | ✅ | — | Usuario de PostgreSQL |
| `DB_PASSWORD` | ✅ | — | Contraseña (se escapa con `quote_plus` para caracteres especiales) |
| `DB_HOST` | ✅ | — | Host del servidor PostgreSQL |
| `DB_PORT` | ❌ | `5432` | Puerto de PostgreSQL |
| `DB_NAME` | ❌ | `postgres` | Nombre de la base de datos |

`quote_plus(DB_PASSWORD)` convierte caracteres especiales como `@` o `#` en la contraseña a formato URL-safe (e.g., `@` → `%40`).

In [2]:
# ============================================
# Carga de variables de entorno para la conexión a PostgreSQL
# ============================================
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT", "5432")
DB_NAME     = os.getenv("DB_NAME", "postgres")

if not all([DB_USER, DB_PASSWORD, DB_HOST]):
    raise ValueError(
        "❌ Faltan variables de base de datos en .env\n"
        "Requeridas: DB_USER, DB_PASSWORD, DB_HOST\n"
        "Opcionales: DB_PORT (default: 5432), DB_NAME (default: postgres)"
    )

# quote_plus maneja caracteres especiales en la contraseña (@ # $ etc.)
DATABASE_URL = f"postgresql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"🔌 Conectando como: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

🔌 Conectando como: postgres.ebdkewopehhsvkhboofb@aws-1-us-east-1.pooler.supabase.com:5432/postgres


## 2. Crear Tabla en PostgreSQL

`PostgresChatMessageHistory` necesita una tabla existente para leer y escribir mensajes.
`create_tables()` la crea si no existe, con la siguiente estructura:

| Columna | Tipo | Descripción |
|---|---|---|
| `id` | `SERIAL` | Clave primaria autoincremental |
| `session_id` | `TEXT` | UUID que identifica la conversación |
| `message` | `JSONB` | Mensaje serializado (tipo + contenido) |
| `created_at` | `TIMESTAMP` | Marca de tiempo del mensaje |

Esta función se llama **una sola vez al iniciar** el agente. Si la tabla ya existe, no hace nada.

In [3]:
# ============================================
# CREAR TABLA DE HISTORIAL de texto en PostgreSQL
# ============================================

# Nombre de la tabla en supabase para almacenar el historial textual de chat User-Bot
TBL_NAME_CHAT_USER_BOT = os.getenv("TBL_NAME_CHAT_USER_BOT")

def crear_tabla_historial(table_name: str = TBL_NAME_CHAT_USER_BOT):
    """Crea la tabla de historial en PostgreSQL si no existe."""
    try:
        sync_connection = psycopg.connect(DATABASE_URL)

        # Se crea tabla (si no existe) con la estructura necesaria 
        # para almacenar el historial de chat (creada por "PostgresChatMessageHistory")
        PostgresChatMessageHistory.create_tables(sync_connection, table_name)
        
        sync_connection.close()

        print(f"✅ Tabla '{table_name}' lista en PostgreSQL")
    except Exception as e:
        print(f"⚠️ Nota sobre tabla: {e}")

crear_tabla_historial()

✅ Tabla 'tbl_chat_history_text' lista en PostgreSQL


## 3. Configuración del Modelo

Sin cambios respecto al agente básico: GPT-4.1 de OpenAI con `temperature=0.7`.

In [ ]:
# ============================================
# Inicializa el modelo de chat con la configuración deseada
# ============================================
chat = init_chat_model(
    "gpt-4.1",
    temperature=0.1,
)

## 4. Prompt con `MessagesPlaceholder` ← diferencia clave

Esta es la **principal diferencia** respecto al agente básico.

### Agente básico (sin memoria)
```python
ChatPromptTemplate.from_messages([
    ("system", "..."),
    ("human", "{input}")        # solo el mensaje actual
])
```

### Este agente (con memoria)
```python
ChatPromptTemplate.from_messages([
    ("system", "..."),
    MessagesPlaceholder("history"),  # ← ranura para el historial
    ("human", "{input}")
])
```

`MessagesPlaceholder(variable_name="history")` es una ranura vacía en el prompt. Cuando se ejecuta la cadena, `RunnableWithMessageHistory` la rellena con la lista de mensajes pasados `[HumanMessage, AIMessage, HumanMessage, AIMessage, ...]` recuperados de PostgreSQL.

Sin este placeholder, el LLM recibiría solo el mensaje actual, sin contexto de la conversación.

In [ ]:
# ============================================
# Se inicializa el prompt de chat con el historial de conversación
# ============================================
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente de IA útil y amigable llamado DataBot.
Responde las preguntas del usuario de manera clara y concisa.
Puedes recordar conversaciones anteriores gracias a tu memoria persistente.
Responde siempre en español."""),
    MessagesPlaceholder(variable_name="history"),  # ← ② HISTORIAL SE INYECTA AQUÍ
    ("human", "{input}")
])

## 4. Cadena Base (LCEL)

La cadena base es idéntica al agente básico: `prompt | chat`.
La diferencia no está aquí, sino en el paso siguiente donde esta cadena se envuelve con `RunnableWithMessageHistory`.

In [ ]:
# ============================================
# Se crea cadena de ejecución que combina el prompt y el modelo de chat
# ============================================
chain = prompt | chat

## 5. Obtener Historial por Sesión ← momento ①: historial LEÍDO

Esta función es el **punto ① del historial**: donde se lee PostgreSQL.

```
RunnableWithMessageHistory
    │
    │ antes de cada invoke...
    ▼
get_session_history(session_id)
    │
    ▼
PostgresChatMessageHistory  →  SELECT * FROM tbl_chat_history_text WHERE session_id = '...'
    │
    ▼
[HumanMessage('hola'), AIMessage('Hola!'), HumanMessage('cómo te llamas?'), ...]
```

**¿Cuándo se llama?** Automáticamente por `RunnableWithMessageHistory` en cada `invoke()`, antes de ejecutar la cadena.

**¿Qué devuelve?** Un objeto `PostgresChatMessageHistory` que:
- Carga todos los mensajes pasados de esa sesión desde PostgreSQL
- Sirve de buffer para agregar nuevos mensajes después del invoke

In [ ]:
# ============================================
# Función para obtener el historial de la sesión desde PostgreSQL
# ============================================
def get_session_history(session_id: str) -> PostgresChatMessageHistory:
    """
    ① HISTORIAL LEÍDO — llamada automáticamente por RunnableWithMessageHistory.
    Retorna un objeto que lee y escribe mensajes de la sesión desde PostgreSQL.
    """
    sync_connection = psycopg.connect(DATABASE_URL)
    return PostgresChatMessageHistory(
        TBL_NAME_CHAT_USER_BOT,                        # nombre de la tabla
        session_id,                      # UUID que identifica la conversación
        sync_connection=sync_connection  # conexión sincrónica a PostgreSQL
    )

## 6. Chain con Memoria Persistente ← momentos ②③④

`RunnableWithMessageHistory` es el **componente central** que orquesta los 4 momentos del historial.
Envuelve la cadena base (`prompt | chat`) y añade gestión automática de historial.

### Flujo completo en cada `invoke()`

```
chain_con_historial.invoke({"input": "hola"}, config={"session_id": "uuid"})
         │
         ├─ ① Llama a get_session_history("uuid")  → lee mensajes de PostgreSQL
         │
         ├─ ② Rellena MessagesPlaceholder("history") con los mensajes leídos
         │         [HumanMessage, AIMessage, HumanMessage, AIMessage, ...]
         │
         ├─ ③ Ejecuta prompt | chat
         │         prompt → [SystemMessage, ...historial..., HumanMessage("hola")]
         │         chat   → llama a GPT-4.1 → devuelve AIMessage
         │
         └─ ④ Guarda automáticamente en PostgreSQL:
                   HumanMessage("hola")    → INSERT INTO tbl_chat_history_text
                   AIMessage("Hola! ...")  → INSERT INTO tbl_chat_history_text
```

### Parámetros clave

| Parámetro | Valor | Significado |
|---|---|---|
| `runnable` | `chain` | La cadena base a envolver |
| `get_session_history` | función | Cómo obtener el historial de una sesión |
| `input_messages_key` | `"input"` | Clave del dict de entrada con el mensaje del usuario |
| `history_messages_key` | `"history"` | Clave del `MessagesPlaceholder` en el prompt |

In [ ]:
# ===========================================
# Se crea un Runnable que combina la cadena de ejecución con el historial de conversación
#===========================================
chain_con_historial = RunnableWithMessageHistory(
    chain,                           # ③ cadena base: prompt | chat
    get_session_history,             # ① función que devuelve el historial por session_id
    input_messages_key="input",      # clave del mensaje del usuario en el dict de entrada
    history_messages_key="history",  # ② clave del MessagesPlaceholder en el prompt
)

/home/marck/Escritorio/python_uv/proyectos_datapath/AI_Engineering_for_Devs/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 7. Función de Chat con Historial

A diferencia del agente básico, ahora recibe un `session_id` para identificar la conversación.
El `config` dict es como `RunnableWithMessageHistory` sabe qué sesión usar:

```python
config={"configurable": {"session_id": session_id}}
```

Sin este `config`, el historial no se puede asociar a ninguna sesión y la llamada falla.

In [ ]:
# ============================================
# Función para enviar un mensaje al agente con historial persistente
# ============================================
def chat_con_agente(mensaje_usuario: str, session_id: str) -> str:
    """
    Envía un mensaje al agente con historial persistente.
    El session_id identifica la conversación en PostgreSQL.
    """
    respuesta = chain_con_historial.invoke(
        {"input": mensaje_usuario},
        config={"configurable": {"session_id": session_id}}  # ← identifica la sesión para cargar el historial de la conversación desde PostgreSQL
    )
    return respuesta.content

## 8. Loop de Conversación

El loop ofrece dos opciones al usuario:

| Opción | Acción | Resultado |
|---|---|---|
| `1` Nueva conversación | Genera `uuid.uuid4()` | El agente empieza sin contexto previo |
| `2` Continuar sesión | El usuario pega un UUID guardado | El agente carga todos los mensajes anteriores de esa sesión |

El UUID es la clave de persistencia: el mismo UUID en sesiones distintas recupera la misma conversación de PostgreSQL.

In [ ]:
# ============================================
# Función principal para ejecutar el agente con historial de conversación
# ============================================
def main():
    print("=" * 60)
    print("🤖 DataBot - Agente CON MEMORIA PERSISTENTE (PostgreSQL)")
    print("=" * 60)

    print("\nOpciones de sesión:")
    print("  1. Nueva conversación")
    print("  2. Continuar sesión existente (pegar UUID)")

    opcion = input("\nElige (1/2): ").strip()

    if opcion == "2":
        session_id = input("\nPega el UUID de la sesión: ").strip()
        try:
            uuid.UUID(session_id)  # valida que sea un UUID real
        except ValueError:
            print("⚠️ UUID inválido. Creando nueva sesión...")
            session_id = str(uuid.uuid4())
    else:
        print("\nCreando nueva sesión...")
        session_id = str(uuid.uuid4())  # nueva sesión con UUID único

    print(f"\n📝 Session ID: {session_id}")
    print("   (Guarda este ID para continuar la conversación después)")
    print("✅ Este agente RECUERDA tus mensajes anteriores")
    print("Escribe 'salir' para terminar.\n")

    print("*" * 60)
    print("💬 Comienza a chatear con DataBot:")
    while True:

        usuario = input("Usuario: ").strip()
        print(f"\n💬 Usuario: {usuario}")

        if usuario.lower() in ["salir", "exit", "quit"]:
            print(f"\n💾 Tu sesión está guardada.")
            print(f"   UUID: {session_id}")
            print("👋 ¡Hasta luego!")
            break

        # Si el usuario no ingresa nada, se ignora y se solicita nuevamente
        if not usuario:
            print("⚠️ Por favor, ingresa un mensaje.")
            continue

        try:
            respuesta = chat_con_agente(usuario, session_id)
            print(f"🤖 DataBot: {respuesta}\n")
        except Exception as e:
            print(f"\n❌ Error: {e}\n")

In [11]:
# ============================================
# EJECUTAR EL AGENTE
# ============================================
main()

🤖 DataBot - Agente CON MEMORIA PERSISTENTE (PostgreSQL)

Opciones de sesión:
  1. Nueva conversación
  2. Continuar sesión existente (pegar UUID)

Creando nueva sesión...

📝 Session ID: 3fcc213a-bad6-4e94-9be7-87bdb489d8ab
   (Guarda este ID para continuar la conversación después)
✅ Este agente RECUERDA tus mensajes anteriores
Escribe 'salir' para terminar.

************************************************************
💬 Comienza a chatear con DataBot:

💬 Usuario: Hola soy Marcos
🤖 DataBot: ¡Hola, Marcos! ¿En qué puedo ayudarte hoy?


💬 Usuario: soy matemático y programo en python
🤖 DataBot: ¡Genial, Marcos! Es estupendo saber que eres matemático y que programas en Python. ¿Te gustaría hablar sobre algún tema específico de matemáticas, necesitas ayuda con algún código en Python, o tienes algún proyecto en mente? ¡Cuéntame cómo puedo ayudarte!


💬 Usuario: cómo me llamo?
🤖 DataBot: Te llamas Marcos. ¿Quieres que recuerde algo más sobre ti?


💬 Usuario: en que programo?
🤖 DataBot: Program

## Registro en PostgreSQL

Estructura real de la tabla `tbl_chat_history_text` (en supabase) con los mensajes guardados por `session_id`:

![Registro de conversación en PostgreSQL](../../images/registro_conversacion.png)

In [12]:
# ============================================
# EJECUTAR EL AGENTE
# ============================================
main()

🤖 DataBot - Agente CON MEMORIA PERSISTENTE (PostgreSQL)

Opciones de sesión:
  1. Nueva conversación
  2. Continuar sesión existente (pegar UUID)

📝 Session ID: 3fcc213a-bad6-4e94-9be7-87bdb489d8ab
   (Guarda este ID para continuar la conversación después)
✅ Este agente RECUERDA tus mensajes anteriores
Escribe 'salir' para terminar.

************************************************************
💬 Comienza a chatear con DataBot:

💬 Usuario: dime qué recuerdas de mí?
🤖 DataBot: Recuerdo que te llamas Marcos, que eres matemático y que programas en Python. Si quieres que recuerde algo más sobre ti, solo dímelo.


💬 Usuario: salir

💾 Tu sesión está guardada.
   UUID: 3fcc213a-bad6-4e94-9be7-87bdb489d8ab
👋 ¡Hasta luego!
